# BAZTA — Train Isolation Forest using CICIoT2023 (Kaggle + T4 GPU)

Trains an Isolation Forest on benign traffic for runtime anomaly detection.

Dataset: UNB CIC IOT 2023 Dataset

In [ ]:
import pandas as pd
import numpy as np
import os, json, joblib, warnings, gc
from tqdm import tqdm
warnings.filterwarnings('ignore')

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [ ]:
# ── Kaggle Paths ──
DATASET_DIRECTORY = '/kaggle/input/unb-cic-iot-2023-dataset/wataiData/csv/CICIoT2023/'
OUTPUT_DIR = '/kaggle/working/'

df_sets = sorted([k for k in os.listdir(DATASET_DIRECTORY) if k.endswith('.csv')])
training_sets = df_sets[:int(len(df_sets)*0.8)]
test_sets = df_sets[int(len(df_sets)*0.8):]
print(f'Total: {len(df_sets)}, Train: {len(training_sets)}, Test: {len(test_sets)}')

In [ ]:
# 4 features mapped for live inference by trust_engine.py
LIVE_FEATURE_COLS = ['Rate', 'Srate', 'Protocol Type', 'Variance']
LIVE_FEATURE_NAMES = ['pkt_rate', 'byte_rate', 'unique_ports', 'port_entropy']
y_column = 'label'

# Preview
sample = pd.read_csv(DATASET_DIRECTORY + training_sets[0], nrows=3)
print('Labels:', sample[y_column].values)
sample.head()

In [ ]:
# Collect benign samples from training files
print('Collecting benign samples...')
benign_chunks = []
total_rows = 0

for f in tqdm(training_sets):
    d = pd.read_csv(DATASET_DIRECTORY + f, usecols=LIVE_FEATURE_COLS + [y_column])
    total_rows += len(d)
    b = d[d[y_column] == 'BenignTraffic'][LIVE_FEATURE_COLS]
    if len(b) > 0:
        benign_chunks.append(b)
    del d; gc.collect()

X_benign = pd.concat(benign_chunks, ignore_index=True)
X_benign.replace([np.inf, -np.inf], np.nan, inplace=True)
X_benign.fillna(0, inplace=True)
del benign_chunks; gc.collect()

print(f'Total rows: {total_rows:,}, Benign: {len(X_benign):,}')

In [ ]:
# Scale and train
live_scaler = StandardScaler()
X_benign_scaled = live_scaler.fit_transform(X_benign)

print('Training Isolation Forest...')
if_model = IsolationForest(
    n_estimators=200,
    max_samples=0.8 if len(X_benign_scaled) > 1000 else 'auto',
    contamination=0.05,
    random_state=42,
    n_jobs=-1,
)
if_model.fit(X_benign_scaled)
print('Training complete!')

In [ ]:
# Evaluate on test set
print('Evaluating...')
y_true_all, y_pred_all = [], []

for f in tqdm(test_sets):
    d = pd.read_csv(DATASET_DIRECTORY + f, usecols=LIVE_FEATURE_COLS + [y_column])
    X_t = d[LIVE_FEATURE_COLS].replace([np.inf, -np.inf], np.nan).fillna(0)
    X_t_scaled = live_scaler.transform(X_t)
    y_true = (d[y_column] != 'BenignTraffic').astype(int)
    y_pred = np.where(if_model.predict(X_t_scaled) == 1, 0, 1)
    y_true_all.extend(y_true.values)
    y_pred_all.extend(y_pred)
    del d; gc.collect()

if_acc = accuracy_score(y_true_all, y_pred_all)
if_prec = precision_score(y_true_all, y_pred_all, zero_division=0)
if_rec = recall_score(y_true_all, y_pred_all, zero_division=0)
if_f1 = f1_score(y_true_all, y_pred_all, zero_division=0)

print(f'\n=== Isolation Forest Results ===')
print(f'Accuracy:  {if_acc:.4f}')
print(f'Precision: {if_prec:.4f}')
print(f'Recall:    {if_rec:.4f}')
print(f'F1 Score:  {if_f1:.4f}')
print(f'\n{classification_report(y_true_all, y_pred_all, target_names=["Benign", "Attack"])}')

In [ ]:
# Save model artifacts to /kaggle/working/
joblib.dump(if_model, os.path.join(OUTPUT_DIR, 'live_if_model.pkl'))
joblib.dump(live_scaler, os.path.join(OUTPUT_DIR, 'live_scaler.pkl'))

meta = {
    'model_type': 'IsolationForest',
    'features': LIVE_FEATURE_COLS,
    'live_feature_names': LIVE_FEATURE_NAMES,
    'best_params': {'n_estimators': 200, 'contamination': 0.05, 'max_samples': 0.8},
    'accuracy': round(if_acc, 4),
    'precision': round(if_prec, 4),
    'recall': round(if_rec, 4),
    'f1_score': round(if_f1, 4),
    'training_samples': int(len(X_benign_scaled)),
    'dataset': 'CICIoT2023 (WATAI)',
    'total_files': len(df_sets),
}
with open(os.path.join(OUTPUT_DIR, 'live_model_meta.json'), 'w') as f:
    json.dump(meta, f, indent=2)

print('\n=== Files saved to /kaggle/working/ ===')
print('  - live_if_model.pkl')
print('  - live_scaler.pkl')
print('  - live_model_meta.json')
print('\nDownload these and place in your project models/ folder.')